# Backpropagation

CSCI 6379 · Topic 15. Backprop is the chain rule applied from the output back to the input. Here we compute the gradients by hand for a single neuron and for a network with a hidden layer, then confirm each with PyTorch autograd.

## The chain rule

If z depends on y and y depends on x, then dz/dx = (dz/dy)(dy/dx). Example: z = (3x^2+2x+1)^2 at x=1.

In [ ]:
x = 1.0
y = 3*x**2 + 2*x + 1
dz_dx = 2*y * (6*x + 2)          # 2(3x^2+2x+1)(6x+2)
print("y =", y, " dz/dx =", dz_dx)   # -> 6.0, 96.0

## Worked example 1: a single sigmoid neuron

W=0.8, b=0.1, x=0.5, target y=1. Forward, then multiply the three local derivatives backward.

In [ ]:
import numpy as np
sigmoid = lambda z: 1/(1+np.exp(-z))

x, W, b, y_true, eta = 0.5, 0.8, 0.1, 1.0, 0.1

# forward
z = W*x + b
yhat = sigmoid(z)
L = 0.5*(yhat - y_true)**2
print(f"forward:  z={z}  yhat={yhat:.4f}  L={L:.4f}")

# backward (chain rule)
dL_dyhat = yhat - y_true              # -0.3775
dyhat_dz = yhat*(1 - yhat)            # sigmoid'(z) = 0.235
dz_dW    = x                          # 0.5
dL_dW = dL_dyhat * dyhat_dz * dz_dW
dL_db = dL_dyhat * dyhat_dz * 1
print(f"backward: dL/dW={dL_dW:.4f}  dL/db={dL_db:.4f}")
print(f"update:   W -> {W - eta*dL_dW:.4f}   b -> {b - eta*dL_db:.4f}")

In [ ]:
# cross-check with PyTorch autograd
import torch
xt = torch.tensor(0.5); yt = torch.tensor(1.0)
W = torch.tensor(0.8, requires_grad=True); b = torch.tensor(0.1, requires_grad=True)
L = 0.5*(torch.sigmoid(W*xt + b) - yt)**2
L.backward()
print(f"autograd: dL/dW={W.grad:.4f}  dL/db={b.grad:.4f}")   # matches -0.0444, -0.0887

## Worked example 2: one hidden layer

x=1 -> hidden neuron (W1=0.5, b1=0) -> output neuron (W2=0.5, b2=0), both sigmoid, y=1. The output error is propagated back through W2 into the hidden layer: dL/dh = delta2 * W2.

In [ ]:
x, W1, b1, W2, b2, y_true = 1.0, 0.5, 0.0, 0.5, 0.0, 1.0

# forward
z1 = W1*x + b1;  h    = sigmoid(z1)
z2 = W2*h + b2;  yhat = sigmoid(z2)
L = 0.5*(yhat - y_true)**2
print(f"forward:  h={h:.4f}  yhat={yhat:.4f}  L={L:.4f}")

# backward: output neuron
delta2 = (yhat - y_true) * yhat*(1 - yhat)      # output error signal
dL_dW2 = delta2 * h
dL_db2 = delta2
# propagate the error back through W2 into the hidden layer
dL_dh  = delta2 * W2
delta1 = dL_dh * h*(1 - h)                       # hidden error signal
dL_dW1 = delta1 * x
dL_db1 = delta1
print(f"output:  delta2={delta2:.4f}  dL/dW2={dL_dW2:.4f}  dL/db2={dL_db2:.4f}")
print(f"propagate: dL/dh = delta2*W2 = {dL_dh:.4f}")
print(f"hidden:  delta1={delta1:.4f}  dL/dW1={dL_dW1:.4f}  dL/db1={dL_db1:.4f}")

In [ ]:
# cross-check with autograd
xt = torch.tensor(1.0); yt = torch.tensor(1.0)
W1 = torch.tensor(0.5, requires_grad=True); b1 = torch.tensor(0.0, requires_grad=True)
W2 = torch.tensor(0.5, requires_grad=True); b2 = torch.tensor(0.0, requires_grad=True)
h  = torch.sigmoid(W1*xt + b1)
yh = torch.sigmoid(W2*h + b2)
L  = 0.5*(yh - yt)**2
L.backward()
print(f"autograd: dL/dW1={W1.grad:.4f} dL/db1={b1.grad:.4f} dL/dW2={W2.grad:.4f} dL/db2={b2.grad:.4f}")

## Autograd on a vector

The general point: PyTorch records the forward operations and runs backprop for you.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x + 2
z = 2 * y * y
out = z.mean()
out.backward()
print("x.grad =", x.grad.tolist())       # 4/3 * y = [4.0, 5.333, 6.667]